In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

epoch = 5000
n_data = 600
n_hidden = 200

x = np.linspace(-15, 15, n_data).reshape(-1, 1)
y = 2 * np.cos(x)

def safe_sigmoid(z):
    z = np.clip(z, -100, 100)
    return 1 / (1 + np.exp(-z))

act_func = [
    lambda z: np.maximum(0, z),
    safe_sigmoid,
    lambda z: np.tanh(np.clip(z, -100, 100)),
    lambda z: z,
    lambda z: np.clip(z, -1, 1),
    lambda z: 1 / (z**2 + 1)
]

fname = {0: "ReLU", 1: "Sigmoid", 2: "Tanh", 3: "Linear", 4: "Clipper-linear", 5: "Rational (1/z²+1)"}

diff = {
    0: lambda z: (z > 0).astype(float),
    1: lambda z: safe_sigmoid(z) * (1 - safe_sigmoid(z)),
    2: lambda z: 1 - np.tanh(np.clip(z, -100, 100))**2,
    3: lambda z: np.ones_like(z),
    4: lambda z: ((z > -1) & (z < 1)).astype(float),
    5: lambda z: (-2 * z) / (z**2 + 1)**2
}

learning_rates = {
    0: 0.000005,
    1: 0.0001,
    2: 0.00005,
    3: 0.000005,
    4: 0.000005,
    5: 0.0001
}

np.random.seed(42)
master_wi1 = np.random.uniform(-0.5, 0.5, (1, n_hidden))
master_bi1 = np.random.uniform(-0.5, 0.5, (1, n_hidden))
master_wi2 = np.random.uniform(-0.5, 0.5, (n_hidden, 1))
master_bi2 = np.random.uniform(-0.5, 0.5, (1, 1))

SAVE_EVERY = 10
results = [[] for _ in range(6)]

for i, fn in enumerate(act_func):
    wi1 = master_wi1.copy()
    wi2 = master_wi2.copy()
    bi1 = master_bi1.copy()
    bi2 = master_bi2.copy()
    lr = learning_rates[i]

    for e in range(epoch):
        ri1 = x @ wi1 + bi1
        fi1 = fn(ri1)
        ri2 = fi1 @ wi2 + bi2

        deltak = np.clip(ri2 - y, -5, 5)
        deltah = np.clip((deltak @ wi2.T) * diff[i](ri1), -5, 5)

        wi2 -= lr * (fi1.T @ deltak)
        bi2 -= lr * np.sum(deltak, axis=0, keepdims=True)
        wi1 -= lr * (x.T @ deltah)
        bi1 -= lr * np.sum(deltah, axis=0, keepdims=True)

        if e % SAVE_EVERY == 0:
            r = fn(x @ wi1 + bi1) @ wi2 + bi2
            results[i].append((e, r.copy()))

def show_animation(idx):
    fig, ax = plt.subplots(figsize=(10, 5), dpi=100) # เพิ่มความคมชัด
    ax.set_xlim(-15, 15)
    ax.set_ylim(-10, 10)
    
    # ปรับเส้น Target ให้จางลงเหมือนในคลิป
    ax.plot(x, y, color='red', linestyle='--', linewidth=1, label='Target', alpha=0.3)
    
    # ปรับเส้น Neural Network ให้หนาและชัดเจน
    line, = ax.plot([], [], color='#1f77b4', lw=2.5, label='Neural Network Prediction')
    
    title = ax.set_title('', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
    ax.grid(True, linestyle=':', alpha=0.6)

    def update(frame):
        ep, y_pred = results[idx][frame]
        line.set_data(x.flatten(), y_pred.flatten())
        title.set_text(f'Activation: {fname[idx]} | Epoch: {ep}')
        return line, title

    anim = FuncAnimation(
        fig, update, 
        frames=len(results[idx]), 
        interval=20, # ปรับความเร็ว (ยิ่งน้อยยิ่งเร็วและสมูท)
        blit=True
    )
    plt.close()
    display(HTML(anim.to_jshtml()))

In [ ]:
wx = np.arange(-10,10,0.1)

## ReLU

In [ ]:
plt.plot(wx,np.array([max(e,0) for e in list(wx)]))

In [ ]:
show_animation(0)

## Sigmoid Unit

In [ ]:
wx = np.arange(-10,10,0.1)
plt.plot(wx,wx>0)
plt.plot(wx,1/(1+np.exp(-wx)))
plt.plot(wx,1/(1+np.exp(-5*wx)))
plt.plot(wx,1/(1+np.exp(-0.5*wx)))

In [ ]:
show_animation(1)


## Tanh

In [ ]:
plt.plot(wx,np.tanh(wx))

In [ ]:
show_animation(2)

## Linear

In [ ]:
plt.plot(wx, wx)

In [ ]:
show_animation(3)

## Clipper-linear

In [ ]:
plt.plot(wx, np.clip(wx, -1, 1))

In [ ]:
show_animation(4)

## Rational (1/z²+1)

In [ ]:
plt.plot(wx, 1 / (wx**2 + 1))

In [ ]:
show_animation(5)

In [ ]:
#